## Audio Preprocessing and Clipping

Reads downloaded manifests, loads audio and build's fixed 3-second windows. Label's gated windows with a BirdNet Teacher model. 

In [1]:
import sys
from pathlib import Path

import librosa
import pandas as pd
from tqdm.auto import tqdm

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "config.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "config.py").exists():
    raise FileNotFoundError(f"Couldn't find src/config.py from {Path.cwd()}")

sys.path.insert(0, str(repo_root))

from src.config import CONFIG
from src.dataset.utils.preprocessing import (
    apply_adaptive_rms_gate_to_windows,
    build_candidate_windows_from_recording,
    build_selection_lookup_for_windows,
    compute_scaled_species_clip_cap,
    create_birdnet_teacher_analyzer,
    ensure_preprocessing_output_dirs,
    label_windows_with_birdnet_teacher,
    list_species_labels_with_downloaded_manifests,
    load_downloaded_manifest_for_species,
    resolve_downloaded_audio_path_from_manifest,
    resolve_preprocessing_paths,
    save_selected_clip_as_wav,
    select_training_windows_per_recording,
    write_species_clip_manifest,
)

cfg = CONFIG.preprocessing


/Users/justyna-przy/Projects/ISE/FYP/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/justyna-przy/Projects/ISE/FYP/.venv/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
paths = resolve_preprocessing_paths(repo_root)
ensure_preprocessing_output_dirs(paths)

print("DATA_DIR:", paths.data_dir)
print("RAW_DIR:", paths.raw_dir)
print("MANIFEST_DIR:", paths.manifest_dir)
print("SPECIES_CLIPS_DIR:", paths.species_clips_dir)
print("NONBIRD_CLIPS_DIR:", paths.nonbird_clips_dir)


DATA_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data
RAW_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data/raw
MANIFEST_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data/manifests
SPECIES_CLIPS_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data/clips/species
NONBIRD_CLIPS_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data/clips/non_bird


### 3) Initialize BirdNET Teacher and Find Downloaded Species

In [3]:
analyzer = create_birdnet_teacher_analyzer()
downloaded_species = list_species_labels_with_downloaded_manifests(paths.manifest_dir)

print("BirdNET analyzer loaded.")
print("Downloaded species:", len(downloaded_species))
downloaded_species[:10]


Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
BirdNET analyzer loaded.
Downloaded species: 50


/Users/justyna-przy/Projects/ISE/FYP/.venv/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


['accipiter_nisus',
 'acrocephalus_schoenobaenus',
 'aegithalos_caudatus',
 'alcedo_atthis',
 'anthus_pratensis',
 'apus_apus',
 'buteo_buteo',
 'carduelis_carduelis',
 'chloris_chloris',
 'cinclus_cinclus']

In [4]:
sample_species = downloaded_species[0]
df_manifest = load_downloaded_manifest_for_species(paths.manifest_dir, sample_species, max_recordings=10)

print("Sample species:", sample_species)
print("Recordings in sample manifest:", len(df_manifest))
df_manifest.head(10)


Sample species: accipiter_nisus
Recordings in sample manifest: 10


,xc_id,sci_name,common_name,recordist,country,date,time,length_s,quality,type,method,animal_seen,also,sampling_rate,xc_url,file_url,local_path
0,1024301,Accipiter nisus,Eurasian Sparrowhawk,David Darrell-Lambert,United Kingdom,2025-07-30,07:00,26,A,call,field recording,yes,[],32000,https://xeno-canto.org/1024301,https://xeno-canto.org/1024301/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
1,1024300,Accipiter nisus,Eurasian Sparrowhawk,David Darrell-Lambert,United Kingdom,2025-07-30,07:00,35,A,call,field recording,yes,[],32000,https://xeno-canto.org/1024300,https://xeno-canto.org/1024300/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
2,1015208,Accipiter nisus,Eurasian Sparrowhawk,David Darrell-Lambert,United Kingdom,2020-07-25,07:30,6,A,call,field recording,yes,[],48000,https://xeno-canto.org/1015208,https://xeno-canto.org/1015208/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
3,1015207,Accipiter nisus,Eurasian Sparrowhawk,David Darrell-Lambert,United Kingdom,2020-07-25,07:30,11,A,call,field recording,yes,[],48000,https://xeno-canto.org/1015207,https://xeno-canto.org/1015207/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
4,1015205,Accipiter nisus,Eurasian Sparrowhawk,David Darrell-Lambert,United Kingdom,2020-07-25,07:30,37,A,call,field recording,yes,[],48000,https://xeno-canto.org/1015205,https://xeno-canto.org/1015205/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
5,916789,Accipiter nisus,Eurasian Sparrowhawk,Andrew Harrop,United Kingdom,2024-06-25,13:30,5,A,"alarm call, call",field recording,yes,[],44100,https://xeno-canto.org/916789,https://xeno-canto.org/916789/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
6,755238,Accipiter nisus,Eurasian Sparrowhawk,Mark Newsome,United Kingdom,2022-10-10,21:00,5,A,nocturnal flight call,field recording,no,[],44100,https://xeno-canto.org/755238,https://xeno-canto.org/755238/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
7,741322,Accipiter nisus,Eurasian Sparrowhawk,Dean McDonnell,United Kingdom,2022-08-01,14:00,35,A,"begging call, call",field recording,no,[],44100,https://xeno-canto.org/741322,https://xeno-canto.org/741322/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
8,738943,Accipiter nisus,Eurasian Sparrowhawk,Dean McDonnell,Ireland,2022-07-10,12:00,4,A,call,field recording,no,[],44100,https://xeno-canto.org/738943,https://xeno-canto.org/738943/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...
9,726430,Accipiter nisus,Eurasian Sparrowhawk,Dean McDonnell,Ireland,2021-05-10,08:00,9,A,call,field recording,no,[],44100,https://xeno-canto.org/726430,https://xeno-canto.org/726430/download,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...


### Preprocessing Routine

In [5]:
def preprocess_species(species_label: str, max_recordings: int | None = None) -> pd.DataFrame:
    manifest_df = load_downloaded_manifest_for_species(
        paths.manifest_dir,
        species_label,
        max_recordings=max_recordings,
    )

    out_rows = []
    downloaded_count = int(len(manifest_df))
    scaled_species_clip_cap = compute_scaled_species_clip_cap(
        downloaded_count,
        baseline_downloaded_count=350,
        baseline_max_species_clips_per_rec=cfg.max_species_clips_per_rec,
        max_scaled_species_clips_per_rec=6,
    )
    print(
        f"{species_label}: downloaded={downloaded_count}, "
        f"max_species_clips_per_rec={scaled_species_clip_cap}"
    )
    out_species_dir = paths.species_clips_dir / species_label
    out_nonbird_dir = paths.nonbird_clips_dir / species_label

    for _, rec in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc=species_label):
        xc_id = str(rec.get("xc_id", ""))
        sci_name = str(rec.get("sci_name", ""))
        target_sci_name_for_teacher = sci_name
        if sci_name.strip().lower() == "coloeus monedula":
            target_sci_name_for_teacher = "Corvus monedula"

        src_path = resolve_downloaded_audio_path_from_manifest(
            rec.get("local_path", ""),
            repo_root=repo_root,
            data_dir=paths.data_dir,
        )

        if src_path is None:
            print(f"Skipping XC{xc_id}: missing/invalid local_path in manifest")
            continue

        # Load full recording as mono 16 kHz
        try:
            full_wave_16k, _ = librosa.load(str(src_path), sr=cfg.sample_rate_model, mono=True)
        except Exception as exc:
            print(f"Skipping XC{xc_id}: failed to load {src_path} ({exc})")
            continue


        candidate_windows = build_candidate_windows_from_recording(
            full_wave_16k,
            sample_rate_model=cfg.sample_rate_model,
            clip_len_s=cfg.clip_len_s,
            stride_s=cfg.stride_s,
            skip_first_s=cfg.skip_first_s,
            rms_eps=cfg.eps,
        )

        gated_windows, gate_thr_db = apply_adaptive_rms_gate_to_windows(
            candidate_windows,
            rms_keep_percentile=cfg.rms_keep_percentile,
            rms_abs_min_db=cfg.rms_abs_min_db,
        )

        labeled_windows = label_windows_with_birdnet_teacher(
            gated_windows,
            target_sci_name=target_sci_name_for_teacher,
            analyzer=analyzer,
            sample_rate_model=cfg.sample_rate_model,
            sample_rate_teacher=cfg.sample_rate_teacher,
            bird_conf_thr=cfg.bird_conf_thr,
            species_conf_thr=cfg.species_conf_thr,
        )

        selected_species, selected_nonbird = select_training_windows_per_recording(
            labeled_windows,
            xc_id=xc_id,
            max_species_clips_per_rec=scaled_species_clip_cap,
            nonbird_clips_per_rec=cfg.nonbird_clips_per_rec,
            seed=cfg.seed,
        )
        selection_lookup = build_selection_lookup_for_windows(selected_species, selected_nonbird)

        for w in labeled_windows:
            key = (float(w["start_s"]), float(w["end_s"]))
            selected_class = selection_lookup.get(key)
            is_selected = int(selected_class is not None)

            clip_path = ""
            selected_reason = ""

            if selected_class == "species":
                clip_path = save_selected_clip_as_wav(
                    w["wave_16k"],
                    out_dir=out_species_dir,
                    xc_id=xc_id,
                    start_s=float(w["start_s"]),
                    end_s=float(w["end_s"]),
                    sample_rate_model=cfg.sample_rate_model,
                )
                selected_reason = "rand_species"
                final_class = "species"
            elif selected_class == "non_bird":
                clip_path = save_selected_clip_as_wav(
                    w["wave_16k"],
                    out_dir=out_nonbird_dir,
                    xc_id=xc_id,
                    start_s=float(w["start_s"]),
                    end_s=float(w["end_s"]),
                    sample_rate_model=cfg.sample_rate_model,
                )
                selected_reason = "nonbird_lowconf"
                final_class = "non_bird"
            else:
                final_class = str(w.get("teacher_decision", "drop"))

            out_rows.append({
                "species_label": species_label,
                "xc_id": xc_id,
                "sci_name": sci_name,
                "source_path": str(src_path),
                "start_s": float(w["start_s"]),
                "end_s": float(w["end_s"]),
                "rms_db": float(w["rms_db"]),
                "rms_gate_thr_db": float(gate_thr_db),
                "teacher_decision": w.get("teacher_decision", ""),
                "teacher_top_sci": w.get("teacher_top_sci", ""),
                "teacher_top_common": w.get("teacher_top_common", ""),
                "teacher_top_conf": float(w.get("teacher_top_conf", 0.0)),
                "teacher_max_conf": float(w.get("teacher_max_conf", 0.0)),
                "final_class": final_class,
                "selected": is_selected,
                "selected_reason": selected_reason,
                "clip_path": clip_path,
                "error": "",
            })

    species_df = write_species_clip_manifest(paths.manifest_dir, species_label, out_rows)
    out_csv = paths.manifest_dir / f"{species_label}_clips.csv"
    print("Wrote:", out_csv)
    return species_df


### 7) Run One Species and Inspect Output

In [6]:
df_sample_clips = preprocess_species(sample_species, max_recordings=5)
df_sample_clips.head(5)


accipiter_nisus: downloaded=5, max_species_clips_per_rec=6


accipiter_nisus: 100%|██████████| 5/5 [00:01<00:00,  4.48it/s]

Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data/manifests/accipiter_nisus_clips.csv


,species_label,xc_id,sci_name,source_path,start_s,end_s,rms_db,rms_gate_thr_db,teacher_decision,teacher_top_sci,teacher_top_common,teacher_top_conf,teacher_max_conf,final_class,selected,selected_reason,clip_path,error
0,accipiter_nisus,1024301,Accipiter nisus,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,1.0,4.0,-25.472088,-26.809049,species,Accipiter nisus,Eurasian Sparrowhawk,0.982468,0.982468,species,1,rand_species,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,
1,accipiter_nisus,1024301,Accipiter nisus,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,4.0,7.0,-26.214006,-26.809049,non_bird,,,0.000000,0.000000,non_bird,1,nonbird_lowconf,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,
2,accipiter_nisus,1024301,Accipiter nisus,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,10.0,13.0,-24.860760,-26.809049,non_bird,,,0.000000,0.000000,non_bird,1,nonbird_lowconf,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,
3,accipiter_nisus,1024301,Accipiter nisus,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,13.0,16.0,-25.913951,-26.809049,non_bird,,,0.000000,0.000000,non_bird,0,,,
4,accipiter_nisus,1024301,Accipiter nisus,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,16.0,19.0,-26.384075,-26.809049,species,Accipiter nisus,Eurasian Sparrowhawk,0.984863,0.984863,species,1,rand_species,/Users/justyna-przy/Projects/ISE/FYP/src/bird_...,


### Run on all Species!

In [7]:
max_species = None
max_recordings_per_species = None

species_to_run = ["coloeus_monedula"]
summaries = []

for species_label in species_to_run:
    species_df = preprocess_species(
        species_label,
        max_recordings=max_recordings_per_species,
    )

    selected = species_df[species_df["selected"] == 1]
    summaries.append({
        "species_label": species_label,
        "selected_total": int(len(selected)),
        "selected_species": int((selected["final_class"] == "species").sum()),
        "selected_nonbird": int((selected["final_class"] == "non_bird").sum()),
        "unique_recordings": int(species_df["xc_id"].nunique()),
    })

summary_df = pd.DataFrame(summaries).sort_values("species_label")
summary_csv = paths.manifest_dir / "clips_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Wrote:", summary_csv)
summary_df.head(20)


coloeus_monedula: downloaded=350, max_species_clips_per_rec=3


coloeus_monedula:  53%|█████▎    | 186/350 [00:19<00:09, 16.98it/s]Note: Illegal Audio-MPEG-Header 0x00000000 at offset 260722.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
coloeus_monedula: 100%|██████████| 350/350 [00:35<00:00,  9.96it/s]

Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data/manifests/coloeus_monedula_clips.csv
Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/bird_data/manifests/clips_summary.csv


,species_label,selected_total,selected_species,selected_nonbird,unique_recordings
0,coloeus_monedula,750,715,35,335
